In [1]:
import numpy as np
import chess
import chess.pgn
from tqdm import tqdm

In [2]:
# datos tomados del train
ELO_MIN = 900
ELO_MAX = 2750
ELO_MEAN = 1551.1 
ELO_STD = 299.6  
ELO_MEDIAN = 1548.0
ELO_Q1 = 1332.0
ELO_Q3 = 1763.0
ELO_IQR = ELO_Q3 - ELO_Q1

In [3]:
def normalizar_elo(elo, metodo):
    """Aplica la normalización elegida."""
    if metodo == "minmax":
        # Rango [0, 1]
        return (elo - ELO_MIN) / (ELO_MAX - ELO_MIN)
    elif metodo == "standard":
        # Centrado en 0, desvío 1 (Z-score)
        return (elo - ELO_MEAN) / ELO_STD
    elif metodo == "robust":
        # Resistente a valores atípicos (outliers)
        return (elo - ELO_MEDIAN) / ELO_IQR
    else: # "none" o error
        return elo

In [4]:
def board_to_bitboard_vector_13(board):
    """Convierte un chess.Board a un vector binario de 832 bits (13 x 64)."""
    piece_types = [
        (chess.PAWN, chess.WHITE), (chess.PAWN, chess.BLACK),
        (chess.ROOK, chess.WHITE), (chess.ROOK, chess.BLACK),
        (chess.KNIGHT, chess.WHITE), (chess.KNIGHT, chess.BLACK),
        (chess.BISHOP, chess.WHITE), (chess.BISHOP, chess.BLACK),
        (chess.QUEEN, chess.WHITE), (chess.QUEEN, chess.BLACK),
        (chess.KING, chess.WHITE), (chess.KING, chess.BLACK),
    ]

    vec = np.zeros(13 * 64, dtype=np.uint8)

    # Llenamos los 12 primeros bitboards
    for i, (piece, color) in enumerate(piece_types):
        bitboard = board.pieces(piece, color)
        for square in bitboard:
            vec[i * 64 + square] = 1

    # Bitboard 13: casillas vacías
    occupied = board.occupied
    for square in chess.SQUARES:
        if not occupied & chess.BB_SQUARES[square]:
            vec[12 * 64 + square] = 1

    return vec

In [14]:
def preprocesar_pgn_a_npz(pgn_filename, out_filename, ply_numbers):
    print(f"Preprocesando {pgn_filename} -> {out_filename}")
    X_list = []
    
    # Creamos listas separadas para cada tipo de Elo
    y_raw = []
    y_minmax = []
    y_standard = []
    y_robust = []
    
    max_ply = max(ply_numbers)
    
    with open(pgn_filename, "r", encoding="utf-8") as f:
        with tqdm(desc="Partidas procesadas") as pbar:
            while True:
                try:
                    game = chess.pgn.read_game(f)
                    if game is None:
                        break 
                    
                    board = game.board()
                    secuencia = []
                    
                    for i, move in enumerate(game.mainline_moves()):
                        board.push(move)
                        ply = i + 1
                        
                        if ply in ply_numbers:
                            secuencia.append(board_to_bitboard_vector_13(board))
                            
                        if ply >= max_ply:
                            break
                    
                    if len(secuencia) == len(ply_numbers):
                        elo = (
                            int(game.headers.get("WhiteElo", 0)) +
                            int(game.headers.get("BlackElo", 0))
                        ) / 2
                        
                        # 1. Guardamos la X en 2D
                        X_list.append(np.array(secuencia, dtype=np.float32))
                        
                        # 2. Calculamos y guardamos TODAS las versiones del Elo
                        y_raw.append(np.float32(elo))
                        y_minmax.append(np.float32(normalizar_elo(elo, "minmax")))
                        y_standard.append(np.float32(normalizar_elo(elo, "standard")))
                        y_robust.append(np.float32(normalizar_elo(elo, "robust")))

                except Exception as e:
                    pass 
                
                finally:
                    pbar.update(1)

    print(f"\nGuardando {len(X_list)} partidas válidas...")
    
    # Guardamos cada lista en su propio array dentro del archivo .npz
    np.savez_compressed(
        out_filename, 
        X=np.array(X_list), 
        y_raw=np.array(y_raw),
        y_minmax=np.array(y_minmax),
        y_standard=np.array(y_standard),
        y_robust=np.array(y_robust),
        plies_guardados=np.array(ply_numbers)
    )
    print("¡Preprocesamiento completado!\n")


In [13]:
plies_totales = [15,16,17,18,20,21,22,23,30,31,32,33,40,41,42,43]
preprocesar_pgn_a_npz("../data/generado/noleak/train.pgn", "../data/generado/only_enc/train.npz", plies_totales)

Preprocesando ../data/generado/noleak/train.pgn -> ../data/generado/only_enc/train.npz


Partidas procesadas: 69887it [02:02, 569.43it/s]



Guardando 53256 partidas válidas...
¡Preprocesamiento completado!

16630


In [15]:
preprocesar_pgn_a_npz("../data/generado/noleak/val.pgn", "../data/generado/only_enc/val.npz", plies_totales)

Preprocesando ../data/generado/noleak/val.pgn -> ../data/generado/only_enc/val.npz


Partidas procesadas: 15202it [00:25, 588.64it/s]



Guardando 11597 partidas válidas...
¡Preprocesamiento completado!



In [16]:
preprocesar_pgn_a_npz("../data/generado/noleak/test.pgn", "../data/generado/only_enc/test.npz", plies_totales)

Preprocesando ../data/generado/noleak/test.pgn -> ../data/generado/only_enc/test.npz


Partidas procesadas: 14914it [00:26, 573.49it/s]



Guardando 11431 partidas válidas...
¡Preprocesamiento completado!

